## 2. 텐서플로로 다변량 선형 회귀 구현하기

In [1]:
import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
tf.random.set_seed(1)

2025-11-28 14:07:46.274841: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-28 14:07:46.343661: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 14:07:47.854635: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0


In [2]:
# 개별 특성을 분리해 선언
x1_train = tf.constant([[73.], [93.], [89.], [96.], [73.]])
x2_train = tf.constant([[80.], [88.], [91.], [98.], [66.]])
x3_train = tf.constant([[75.], [93.], [90.], [100.], [70.]])
y_train = tf.constant([[152.], [185.], [180.], [196.], [142.]])
w1 = tf.Variable(tf.zeros([1, 1]))
w2 = tf.Variable(tf.zeros([1, 1]))
w3 = tf.Variable(tf.zeros([1, 1]))
b = tf.Variable(tf.zeros([1]))
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5)
nb_epochs = 10000
for epoch in range(nb_epochs + 1):
    with tf.GradientTape() as tape:
        hypothesis = x1_train * w1 + x2_train * w2 + x3_train * w3 + b
        cost = tf.reduce_mean(tf.square(hypothesis - y_train))
    grads = tape.gradient(cost, [w1, w2, w3, b])
    optimizer.apply_gradients(zip(grads, [w1, w2, w3, b]))
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}/{nb_epochs} w1: {w1.numpy().item():.3f} w2: {w2.numpy().item():.3f} w3: {w3.numpy().item():.3f} b: {b.numpy().item():.3f} Cost: {cost.numpy():.6f}')

Epoch    0/10000 w1: 0.294 w2: 0.294 w3: 0.297 b: 0.003 Cost: 29661.800781


2025-11-28 14:07:49.242481: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch  100/10000 w1: 0.674 w2: 0.661 w3: 0.676 b: 0.008 Cost: 1.563634
Epoch  200/10000 w1: 0.679 w2: 0.655 w3: 0.677 b: 0.008 Cost: 1.497603
Epoch  300/10000 w1: 0.684 w2: 0.649 w3: 0.677 b: 0.008 Cost: 1.435026
Epoch  400/10000 w1: 0.689 w2: 0.643 w3: 0.678 b: 0.008 Cost: 1.375730
Epoch  500/10000 w1: 0.694 w2: 0.638 w3: 0.678 b: 0.009 Cost: 1.319503
Epoch  600/10000 w1: 0.699 w2: 0.633 w3: 0.679 b: 0.009 Cost: 1.266215
Epoch  700/10000 w1: 0.704 w2: 0.627 w3: 0.679 b: 0.009 Cost: 1.215693
Epoch  800/10000 w1: 0.709 w2: 0.622 w3: 0.679 b: 0.009 Cost: 1.167821
Epoch  900/10000 w1: 0.713 w2: 0.617 w3: 0.680 b: 0.009 Cost: 1.122419
Epoch 1000/10000 w1: 0.718 w2: 0.613 w3: 0.680 b: 0.009 Cost: 1.079375
Epoch 1100/10000 w1: 0.722 w2: 0.608 w3: 0.680 b: 0.009 Cost: 1.038569
Epoch 1200/10000 w1: 0.727 w2: 0.603 w3: 0.681 b: 0.010 Cost: 0.999893
Epoch 1300/10000 w1: 0.731 w2: 0.599 w3: 0.681 b: 0.010 Cost: 0.963217
Epoch 1400/10000 w1: 0.735 w2: 0.595 w3: 0.681 b: 0.010 Cost: 0.928421
Epoch 

## 4. 행렬 연산을 사용해 텐서플로로 구현하기

In [3]:
x_train = tf.constant([[73., 80., 75.],
                       [93., 88., 93.],
                       [89., 91., 90.],
                       [96., 98., 100.],
                       [73., 66., 70.]], dtype=tf.float32)
y_train = tf.constant([[152.], [185.], [180.], [196.], [142.]], dtype=tf.float32)

print(x_train.shape)
print(y_train.shape)

W = tf.Variable(tf.zeros([3, 1]))
b = tf.Variable(tf.zeros([1]))
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5)
nb_epochs = 10000

for epoch in range(nb_epochs + 1):
    with tf.GradientTape() as tape:
        hypothesis = tf.matmul(x_train, W) + b
        cost = tf.reduce_mean(tf.square(hypothesis - y_train))
    grads = tape.gradient(cost, [W, b])
    optimizer.apply_gradients(zip(grads, [W, b]))
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}/{nb_epochs} Weights: {tf.squeeze(W).numpy()} Cost: {cost.numpy():.6f}')

with tf.device(None):
    new_input = tf.constant([[75., 85., 72.]], dtype=tf.float32)
    prediction = tf.matmul(new_input, W) + b
    print(f'Predicted value for input {new_input.numpy().squeeze().tolist()}: {float(prediction.numpy())}')

(5, 3)
(5, 1)
Epoch    0/10000 Weights: [0.294012 0.2936   0.29738 ] Cost: 29661.800781
Epoch  100/10000 Weights: [0.67350656 0.66096807 0.67623144] Cost: 1.563634
Epoch  200/10000 Weights: [0.6789459  0.6549617  0.67678434] Cost: 1.497608
Epoch  300/10000 Weights: [0.6842513  0.6491219  0.67730516] Cost: 1.435026
Epoch  400/10000 Weights: [0.68942624 0.6434438  0.6777953 ] Cost: 1.375730
Epoch  500/10000 Weights: [0.69447404 0.6379234  0.67825526] Cost: 1.319511
Epoch  600/10000 Weights: [0.69939834 0.6325562  0.67868614] Cost: 1.266222
Epoch  700/10000 Weights: [0.70420223 0.6273384  0.6790884 ] Cost: 1.215696
Epoch  800/10000 Weights: [0.708889   0.62226564 0.6794635 ] Cost: 1.167818
Epoch  900/10000 Weights: [0.7134622  0.6173338  0.67981136] Cost: 1.122429
Epoch 1000/10000 Weights: [0.7179243  0.61253947 0.68013346] Cost: 1.079378
Epoch 1100/10000 Weights: [0.7222781 0.6078793 0.6804302] Cost: 1.038584
Epoch 1200/10000 Weights: [0.72652686 0.60334903 0.6807024 ] Cost: 0.999894
Epo

/tmp/ipykernel_7702/3580625841.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f'Predicted value for input {new_input.numpy().squeeze().tolist()}: {float(prediction.numpy())}')
